# Compare Segmentation Models

Quick notebook runner for the `topology` segmentation variant on real 64-cube patches. The topology model receives PH features computed from grayscale/raw cubes; binary-derived topology is used only as an auxiliary target.

In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path
from types import SimpleNamespace

import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "utils").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from utils import (  # noqa: E402
    BCEDiceLoss,
    BereaPatchDataset,
    CubeSizeBatchSampler,
    TOPOLOGY_FEATURE_DIM,
    TopologyAdaptiveRoutedUNet3D,
    auxiliary_physics_loss,
    dice_score_from_logits,
    topology_prediction_loss,
)

ROOT

## Config

Keep this small for iteration.

In [ ]:
CUBE_SIZE = 64
EPOCHS = 100
SAMPLES_PER_GROUP = 4
MAX_TRAIN_BATCHES = 8
MAX_VAL_BATCHES = 4
BATCH_SIZE = 1
NUM_WORKERS = 0
BASE_CHANNELS = 8
CTX_DIM = 32
LR = 1.0e-4
AUX_WEIGHT = 0.05
TOPOLOGY_WEIGHT = 0.01
TOPOLOGY_MAX_SIZE = 32
TOPOLOGY_CACHE_DIR = ROOT / "outputs" / "topology_cache"
OUTPUT_CSV = ROOT / "outputs" / f"notebook_segmentation_model_runs_{CUBE_SIZE}.csv"
CHECKPOINT_DIR = ROOT / "models" / "notebook_runs"
AMP = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

print(f"device={DEVICE} root={ROOT}")

In [ ]:
def build_model() -> torch.nn.Module:
    return TopologyAdaptiveRoutedUNet3D(
        base_channels=BASE_CHANNELS,
        ctx_dim=CTX_DIM,
        ph_dim=TOPOLOGY_FEATURE_DIM,
        topology_dim=TOPOLOGY_FEATURE_DIM,
    )


def make_loader(dataset, *, shuffle: bool) -> DataLoader:
    sampler = CubeSizeBatchSampler(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, seed=42 if shuffle else 43)
    kwargs = {"batch_sampler": sampler, "num_workers": NUM_WORKERS, "pin_memory": PIN_MEMORY}
    if NUM_WORKERS > 0:
        kwargs.update({"persistent_workers": True, "prefetch_factor": 2})
    return DataLoader(dataset, **kwargs)


def make_datasets():
    common = dict(
        root_dir=ROOT,
        cube_size=[CUBE_SIZE],
        samples_per_group=SAMPLES_PER_GROUP,
        return_topology=True,
        topology_cache_dir=TOPOLOGY_CACHE_DIR,
        topology_max_size=TOPOLOGY_MAX_SIZE,
    )
    train_ds = BereaPatchDataset(split="train", balance=True, **common)
    val_ds = BereaPatchDataset(split="val", balance=False, noise_types=["none"], **common)
    return train_ds, val_ds


def router_entropy(alpha: torch.Tensor) -> torch.Tensor:
    probs = alpha.float().clamp_min(1.0e-8)
    return -(probs * probs.log()).sum(dim=-1).mean()


def run_epoch(model, loader, criterion, optimizer, scaler, *, train: bool, max_batches: int | None):
    model.train(train)
    totals = {}
    counts = {}
    iterator = tqdm(loader, desc="train" if train else "val", leave=False)

    def update(name, value, n):
        totals[name] = totals.get(name, 0.0) + float(value) * n
        counts[name] = counts.get(name, 0) + n

    for batch_idx, batch in enumerate(iterator):
        if max_batches is not None and batch_idx >= max_batches:
            break

        x = batch["x"].to(DEVICE, non_blocking=True)
        y = batch["y"].to(DEVICE, non_blocking=True)
        porosity = batch["porosity"].to(DEVICE, non_blocking=True)
        percolates = batch["percolates"].to(DEVICE, non_blocking=True)
        ph_features = batch.get("ph_features")
        topology_target = batch.get("topology_target")
        if ph_features is not None:
            ph_features = ph_features.to(DEVICE, non_blocking=True)
        if topology_target is not None:
            topology_target = topology_target.to(DEVICE, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train), torch.amp.autocast(device_type=DEVICE.type, enabled=AMP and DEVICE.type == "cuda"):
            out = model(x, ph_features=ph_features, return_dict=True)
            logits = out["logits"]
            seg_loss, bce_loss, dice_loss = criterion(logits, y)
            aux_loss, _ = auxiliary_physics_loss(
                out,
                y,
                porosity_target=porosity,
                percolation_target=percolates,
                porosity_weight=AUX_WEIGHT,
                percolation_weight=AUX_WEIGHT,
            )
            topo_loss, topo_parts = topology_prediction_loss(out, topology_target, topology_weight=TOPOLOGY_WEIGHT)
            loss = seg_loss + aux_loss + topo_loss

        if train:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        n = x.size(0)
        with torch.no_grad():
            dice = dice_score_from_logits(logits, y)
        update("loss", loss.detach().cpu(), n)
        update("seg_loss", seg_loss.detach().cpu(), n)
        update("aux_loss", aux_loss.detach().cpu(), n)
        update("topology_loss", topo_loss.detach().cpu(), n)
        update("bce", bce_loss.detach().cpu(), n)
        update("dice_loss", dice_loss.detach().cpu(), n)
        update("dice", dice.detach().cpu(), n)
        update("router_entropy", router_entropy(out["router_alpha"]).detach().cpu(), n)
        if "topology_loss" in topo_parts:
            update("topology_loss_raw", topo_parts["topology_loss"].detach().cpu(), n)

        iterator.set_postfix({"loss": f"{totals['loss'] / counts['loss']:.4f}", "dice": f"{totals['dice'] / counts['dice']:.4f}"})

    return {name: totals[name] / counts[name] for name in totals}

## Run Topology Model

Trains the topology-adaptive segmentation model and saves checkpoints under `models/notebook_runs/`.

In [ ]:
all_rows = []

print(f"\n=== topology ===")
train_ds, val_ds = make_datasets()
print("train groups:")
display(train_ds.df.groupby(["rock", "cube_size", "split"]).size().rename("samples").reset_index())

train_loader = make_loader(train_ds, shuffle=True)
val_loader = make_loader(val_ds, shuffle=False)
model = build_model().to(DEVICE)
criterion = BCEDiceLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1.0e-4)
scaler = torch.amp.GradScaler(DEVICE.type, enabled=AMP and DEVICE.type == "cuda")

best_val = float("inf")
checkpoint_path = CHECKPOINT_DIR / f"topology_quick_{CUBE_SIZE}.pth"
start = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        scaler,
        train=True,
        max_batches=MAX_TRAIN_BATCHES,
    )
    val_metrics = run_epoch(
        model,
        val_loader,
        criterion,
        optimizer,
        scaler,
        train=False,
        max_batches=MAX_VAL_BATCHES,
    )
    row = {
        "model": "topology",
        "epoch": epoch,
        "cube_size": CUBE_SIZE,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }
    all_rows.append(row)

    if val_metrics["loss"] < best_val:
        best_val = val_metrics["loss"]
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "model": "topology",
                "epoch": epoch,
                "val_loss": val_metrics["loss"],
                "val_dice": val_metrics["dice"],
                "base_channels": BASE_CHANNELS,
                "ctx_dim": CTX_DIM,
                "ph_dim": TOPOLOGY_FEATURE_DIM,
                "topology_dim": TOPOLOGY_FEATURE_DIM,
            },
            checkpoint_path,
        )

elapsed = time.perf_counter() - start
all_rows[-1]["elapsed_sec"] = elapsed
all_rows[-1]["checkpoint"] = str(checkpoint_path)
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

summary = pd.DataFrame(all_rows)
summary.to_csv(OUTPUT_CSV, index=False)
summary

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title in zip(
    axes,
    ["val_loss", "val_dice"],
    ["Validation Loss", "Validation Dice Score"],
):
    df = summary[summary["model"] == "topology"]
    ax.plot(df["epoch"], df[metric], label="topology", linewidth=1.5)
    ax.set_xlabel("Epoch")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

# Лучший результат
best = summary.loc[summary["val_dice"].idxmax()]
print(f"Best topology: epoch={best['epoch']}, val_loss={best['val_loss']:.4f}, val_dice={best['val_dice']:.4f}")